# Inflow vs. Outflow — All Storm Durations (Neuer Teich, With Reservoir)

**Purpose:** Overlays inflow and outflow hydrographs on the same axes for all
storm durations, to show the reservoir's attenuation effect across the full
range of KOSTRA design events.

**What it does:**
- Loads TALSIM results for the 'with reservoir' configuration
- Produces a 4-panel figure (same layout as Figure Group 1):
  - (a) Inflow  | T = 500yr  — all storm durations
  - (b) Inflow  | T = 5000yr — all storm durations
  - (c) Outflow | T = 500yr  — all storm durations
  - (d) Outflow | T = 5000yr — all storm durations
- Solid lines = inflow, dashed lines = outflow; same color per duration

**User settings:** Edit only the USER SETTINGS block at the top  
**Input:** TALSIM `.WEL` output folder (with reservoir)  
**Output:** 4-panel inflow vs. outflow PNG

---

In [ ]:
# =============================================================================
# FIGURE — Neuer Teich With Reservoir: Inflow vs Outflow
# 4-panel figure — same layout as Figure Group 1:
#   (a) Inflow  T = 500yr  — all storm durations
#   (b) Inflow  T = 5000yr — all storm durations
#   (c) Outflow T = 500yr  — all storm durations
#   (d) Outflow T = 5000yr — all storm durations
# Solid = Inflow, Dashed = Outflow, same color per duration
#
# HOW TO USE:
#   Edit ONLY the USER SETTINGS block below, then run.
# =============================================================================

# %% Imports
from pathlib import Path
import re
import unicodedata
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

# =============================================================================
# *** USER SETTINGS — EDIT ONLY HERE ***
# =============================================================================

# Folder containing all event subfolders (NNN_Xh_Yyr)
MAIN_FOLDER = Path(r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_r\Neuer_Teich_r_100_60min")

# WEL file name inside each event subfolder
WEL_NAME = "Neuer_Teich_r.WEL"

# Inflow column  — total inflow into the reservoir
COL_INFLOW  = "T001_1ZU"

# Outflow column — outflow from the reservoir
COL_OUTFLOW = "S005_1ZU"

# Two return periods (left column | right column)
T1 = 500
T2 = 5000

# Line widths
LW_CRITICAL = 1
LW_NORMAL   = 0.9

# Output folder
OUT_DIR = Path(r"C:\Users\raah\Desktop\Project_ZR\figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Fixed settings
# =============================================================================

re_folder = re.compile(
    r"^(?P<num>\d{3})_(?P<dauer>\d+(?:[.,]\d+)?)h_(?P<yr>\d+)yr$",
    re.IGNORECASE
)

SKIP_DURATIONS = {0.25, 0.5}

# =============================================================================
# %% Helpers
# =============================================================================

def ascii_safe(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

def merge_split_underscore_cols(cols):
    fixed, i = [], 0
    while i < len(cols):
        if (i + 1 < len(cols)
                and cols[i + 1].startswith("_")
                and re.match(r"^[A-Za-z0-9]+$", cols[i])):
            fixed.append(cols[i] + cols[i + 1]); i += 2
        else:
            fixed.append(cols[i]); i += 1
    return fixed

_date_re = re.compile(r"^\d{2}\.\d{2}\.\d{4}$")
_time_re = re.compile(r"^\d{2}:\d{2}$")

def _find_wel(folder, wel_name):
    for name in [wel_name, wel_name.lower(), wel_name + ".txt"]:
        p = folder / name
        if p.exists():
            return p
    raise FileNotFoundError(f"WEL not found in {folder}")

def parse_wel(path, value_col):
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Datum_Zeit"):
            header_idx = i; break
    if header_idx is None:
        raise ValueError(f"Header not found in {path}")
    cols = merge_split_underscore_cols(lines[header_idx].split())
    if value_col not in cols:
        raise KeyError(
            f"Column '{value_col}' not in {path.name}\n"
            f"Available: {[c for c in cols if '1ZU' in c or '1AB' in c]}"
        )
    vpos            = cols.index(value_col)
    expected_tokens = len(cols) + 1
    start = header_idx + 1
    while start < len(lines):
        s = lines[start].strip()
        if not s or s.startswith("*") or s.lstrip().startswith("-"):
            start += 1; continue
        if lines[start].split() and _date_re.match(lines[start].split()[0]):
            break
        start += 1

    def find_dt(buf, at=0):
        for j in range(at, len(buf) - 1):
            if _date_re.match(buf[j]) and _time_re.match(buf[j + 1]):
                return j
        return None

    times, values, buf = [], [], []
    for line in lines[start:]:
        if not line.strip() or line.strip().startswith("*"):
            continue
        buf.extend(line.split())
        while True:
            s0 = find_dt(buf)
            if s0 is None:
                if len(buf) > 10 * expected_tokens: buf = buf[-expected_tokens:]
                break
            if s0 > 0: buf = buf[s0:]
            s1 = find_dt(buf, 2)
            end = s1 if (s1 and s1 < expected_tokens) else expected_tokens
            if len(buf) < end: break
            rec = buf[:end]; buf = buf[end:]
            if len(rec) != expected_tokens: continue
            try:
                ts = dt.datetime.strptime(rec[0] + " " + rec[1], "%d.%m.%Y %H:%M")
            except Exception:
                ts = None
            try:
                val = float(rec[2:][vpos - 1].replace(",", "."))
            except Exception:
                continue
            times.append(ts); values.append(val)

    q = np.asarray(values, dtype=float)
    if times and all(t is not None for t in times):
        t0  = times[0]
        t_h = np.array([(t - t0).total_seconds() / 3600 for t in times])
    else:
        t_h = np.arange(len(q), dtype=float)
    return {"q": q, "t_h": t_h}


def load_both_cols(root, wel_name, col_in, col_out, return_period):
    """
    Load inflow AND outflow for all durations of a given return period.
    Returns { dauer_h: {"inflow": {q,t_h}, "outflow": {q,t_h}, "qmax_in": float} }
    """
    results = {}
    for sf in sorted(root.iterdir()):
        if not sf.is_dir(): continue
        m = re_folder.match(sf.name)
        if not m: continue
        if int(m.group("yr")) != return_period: continue
        dauer_h = float(m.group("dauer").replace(",", "."))
        if dauer_h in SKIP_DURATIONS:
            continue
        try:
            wel = _find_wel(sf, wel_name)
            p_in  = parse_wel(wel, col_in)
            p_out = parse_wel(wel, col_out)
            if len(p_in["q"]) > 0:
                results[dauer_h] = {
                    "inflow":  p_in,
                    "outflow": p_out,
                    "qmax_in": float(np.nanmax(p_in["q"])),
                }
        except Exception as e:
            print(f"  [SKIP] {sf.name}: {e}")
    if not results:
        print(f"  [WARNING] No data for T={return_period}yr")
    return results


# =============================================================================
# %% Color ramp (same peak-ranked scheme as Fig Group 1)
# =============================================================================

def make_colors(dauer_list, data):
    if not dauer_list:
        return {}, None
    critical = max(dauer_list, key=lambda d: data[d]["qmax_in"])
    non_crit = [d for d in dauer_list if d != critical]
    nc       = len(non_crit)
    RAMP = mcolors.LinearSegmentedColormap.from_list("hydro_ramp", [
        "#08306b", "#2196c8", "#1a6b2f",
        "#78c44a", "#f5e400", "#f57c00", "#7b3a10",
    ])
    colors = {d: RAMP(i / max(nc - 1, 1)) for i, d in enumerate(non_crit)}
    colors[critical] = "#e00000"
    return colors, critical


# =============================================================================
# %% Panel plotters
# =============================================================================

def plot_panel_all(ax, data, return_period, panel_title, show_outflow_only=False):
    """
    All durations on one panel.
    show_outflow_only=False → plot inflow (solid lines)
    show_outflow_only=True  → plot outflow (dashed lines)
    Critical duration always in red and thicker.
    """
    dauer_list = sorted(data.keys(), key=float)
    if not dauer_list:
        ax.set_title(f"{panel_title}\n(no data)", pad=7); return

    colors, critical = make_colors(dauer_list, data)

    if show_outflow_only:
        qmax_global = max(np.nanmax(data[d]["outflow"]["q"])
                          for d in dauer_list if len(data[d]["outflow"]["q"]) > 0)
    else:
        qmax_global = max(np.nanmax(data[d]["inflow"]["q"])
                          for d in dauer_list if len(data[d]["inflow"]["q"]) > 0)

    for d in dauer_list:
        is_crit = (d == critical)
        lw  = LW_CRITICAL if is_crit else LW_NORMAL
        al  = 1.0 if is_crit else 0.80
        lbl = f"{d:g} h ★" if is_crit else f"{d:g} h"

        if show_outflow_only:
            q = data[d]["outflow"]["q"]
            t = data[d]["outflow"]["t_h"]
            ls = "-"
        else:
            q = data[d]["inflow"]["q"]
            t = data[d]["inflow"]["t_h"]
            ls = "-"

        ax.plot(t, q, color=colors[d], linestyle=ls,
                linewidth=lw, alpha=al, label=lbl,
                zorder=50 if is_crit else 2)

        # Peak triangle
        idx = int(np.nanargmax(q))
        ax.plot(t[idx], q[idx], marker="^",
                markersize=5 if is_crit else 3.5,
                color=colors[d],
                markeredgewidth=0.5, markeredgecolor="white",
                zorder=60 if is_crit else 5)

    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=qmax_global * 1.15)
    ax.xaxis.set_major_locator(MaxNLocator(8, integer=False))
    ax.tick_params(axis="both", length=4)
    ax.set_xlabel("Time [h] (from event start)", fontsize=8)
    ax.set_ylabel("Discharge [m³/s]", fontsize=8)
    ax.set_title(panel_title, pad=7, fontsize=9, loc="center")

    leg = ax.legend(
        title="Storm duration",
        ncol=2, loc="upper right",
        framealpha=0.92, edgecolor="0.7",
        borderpad=0.5, labelspacing=0.25,
        handlelength=1.6, fontsize=7,
    )
    leg.get_title().set_fontsize(7)
    leg.get_title().set_fontstyle("italic")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_panel_critical(ax, data, return_period, panel_title):
    """
    Critical duration only: inflow (solid) vs outflow (dashed).
    Shaded area between = retained volume.
    """
    dauer_list = sorted(data.keys(), key=float)
    if not dauer_list:
        ax.set_title(f"{panel_title}\n(no data)", pad=7); return

    _, critical = make_colors(dauer_list, data)
    d    = critical
    q_in  = data[d]["inflow"]["q"]
    q_out = data[d]["outflow"]["q"]
    t_in  = data[d]["inflow"]["t_h"]
    t_out = data[d]["outflow"]["t_h"]

    qmax_global = max(np.nanmax(q_in), np.nanmax(q_out))

    # Inflow
    ax.plot(t_in,  q_in,  color="#08306b", linestyle="-",
            linewidth=LW_CRITICAL, alpha=0.95, label=f"Inflow  ({d:g} h ★)")
    # Outflow
    ax.plot(t_out, q_out, color="#c0392b", linestyle="--",
            linewidth=LW_CRITICAL, alpha=0.95, label=f"Outflow ({d:g} h ★)")

    # Shaded retention area
    t_common  = np.union1d(t_in, t_out)
    q_in_i    = np.interp(t_common, t_in,  q_in)
    q_out_i   = np.interp(t_common, t_out, q_out)
    ax.fill_between(
        t_common, q_in_i, q_out_i,
        where=(q_in_i >= q_out_i),
        alpha=0.13, color="#08306b",
        label="Retained volume"
    )

    # Peak markers
    ax.plot(t_in[np.nanargmax(q_in)],   np.nanmax(q_in),
            marker="^", markersize=5, color="#08306b",
            markeredgewidth=0.5, markeredgecolor="white", zorder=10)
    ax.plot(t_out[np.nanargmax(q_out)], np.nanmax(q_out),
            marker="^", markersize=5, color="#c0392b",
            markeredgewidth=0.5, markeredgecolor="white", zorder=10)

    # Peak attenuation annotation
    reduction = np.nanmax(q_in) - np.nanmax(q_out)
    pct       = 100 * reduction / np.nanmax(q_in) if np.nanmax(q_in) > 0 else 0
    ax.annotate(
        f"ΔQmax = {reduction:.2f} m³/s  ({pct:.1f}% attenuation)",
        xy=(0.97, 0.97), xycoords="axes fraction",
        fontsize=7, ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                  edgecolor="0.6", alpha=0.93, linewidth=0.7)
    )

    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=qmax_global * 1.18)
    ax.xaxis.set_major_locator(MaxNLocator(8, integer=False))
    ax.tick_params(axis="both", length=4)
    ax.set_xlabel("Time [h] (from event start)", fontsize=8)
    ax.set_ylabel("Discharge [m³/s]", fontsize=8)
    ax.set_title(panel_title, pad=7, fontsize=9, loc="center")

    leg = ax.legend(
        loc="upper right", framealpha=0.92, edgecolor="0.7",
        borderpad=0.5, labelspacing=0.3,
        handlelength=1.6, fontsize=7,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# =============================================================================
# %% Main
# =============================================================================

plt.rcParams.update({
    "font.family":    "monospace",
    "font.size":       8,
    "axes.titlesize": 9,
    "axes.labelsize":  8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "axes.linewidth":  0.7,
    "grid.linewidth":  0.45,
    "grid.alpha":      0.35,
    "grid.linestyle":  "--",
    "axes.grid":       True,
    "axes.axisbelow":  True,
    "figure.dpi":      150,
})

print("=" * 60)
print("Neuer Teich — Inflow vs Outflow")
print("=" * 60)

print(f"\nLoading T = {T1} yr ...")
data_T1 = load_both_cols(MAIN_FOLDER, WEL_NAME, COL_INFLOW, COL_OUTFLOW, T1)

print(f"Loading T = {T2} yr ...")
data_T2 = load_both_cols(MAIN_FOLDER, WEL_NAME, COL_INFLOW, COL_OUTFLOW, T2)

# 4-panel figure
#   (a) all durations T1  | (b) all durations T2
#   (c) critical only T1  | (d) critical only T2
FIG_WIDTH  = 6.30
FIG_HEIGHT = 5.50
fig, axes  = plt.subplots(2, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

panel_labels = [["(a)", "(b)"], ["(c)", "(d)"]]
for r in range(2):
    for c in range(2):
        axes[r, c].text(-0.08, 1.06, panel_labels[r][c],
                        transform=axes[r, c].transAxes,
                        fontsize=9, fontweight="bold", va="bottom", ha="left")

print("\nPlotting panels...")

# Layout:  (a) Inflow  T1  | (b) Outflow T1
#           (c) Inflow  T2  | (d) Outflow T2
plot_panel_all(axes[0, 0], data_T1, T1, f"Inflow  |  T = {T1} yr")
plot_panel_all(axes[0, 1], data_T1, T1, f"Outflow  |  T = {T1} yr", show_outflow_only=True)
plot_panel_all(axes[1, 0], data_T2, T2, f"Inflow  |  T = {T2} yr")
plot_panel_all(axes[1, 1], data_T2, T2, f"Outflow  |  T = {T2} yr", show_outflow_only=True)

fig.tight_layout()
fig.subplots_adjust(hspace=0.42, wspace=0.32)

outpng = OUT_DIR / "Fig_NT_Inflow_vs_Outflow.png"
fig.savefig(outpng, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"\nSaved: {outpng}")
print("In Word: Insert > Pictures > set width = 16.0 cm")

# =============================================================================
# %% COLUMN HELPER — uncomment to check available columns
# =============================================================================
#
# for sf in sorted(MAIN_FOLDER.iterdir()):
#     if sf.is_dir():
#         wel = sf / WEL_NAME
#         if wel.exists():
#             lines = wel.read_text(encoding="utf-8", errors="replace").splitlines()
#             for line in lines:
#                 if line.strip().startswith("Datum_Zeit"):
#                     cols = line.split()
#                     print("1ZU (inflow) :", [c for c in cols if "1ZU" in c])
#                     print("1AB (outflow):", [c for c in cols if "1AB" in c])
#                     print("WSP (level)  :", [c for c in cols if "WSP" in c])
#                     print("VOL (volume) :", [c for c in cols if "VOL" in c])
#                     break
#         break